# Step 1.4.1 Classifiy and Persist Candidates

Next step in the workflow

Step 5.1 — Improve candidate filtering
Identify likely HRS variables and obvious false positives, such as author names, without deleting source evidence.
Next task

* T he candidate extraction process is capturing many ordinary Codebook terms, not just HRS variable names. 
* We should refine the candidates before moving on to populating the metadata catalog.
* For example:
    - HRS, RAND, and AHEAD are document or study terms.
    - NR, DK, and RF can refer to survey response or missing-data codes, depending on context.
    - W1 through W16 appear to be wave references, but they are not necessarily variables.
    - AGE, HEALTH, INCOME, and PENSION are meaningful subject terms, but a token alone does not establish that it is a valid variable.
    - ADL, IADL, SSI, and SSDI are meaningful abbreviations that may appear in definitions or variable documentation.

Step 5.2 — Review Codebook evidence
Inspect candidate definitions, construction rules, and source pages. Naming patterns alone cannot verify a variable.

Step 5.3 — Record approval decisions
Update review_status to Approved or Rejected, and retain reviewer notes and audit information.

Step 6 — Populate approved metadata
Use approved records to populate hrs_variable_catalog, then process wave coverage, cohort coverage, descriptive statistics, and value labels separately.


### Classify Candidates and update Review Table

This cell applies the preliminary naming pattern and writes the classification to the Variable Review table using a Delta MERGE.

Preliminary classification and persistence. Verify before rerunning.

The corrected cell needs to do three things:
1. Classify candidate records using your preliminary naming pattern.
2. Persist the classifications to dev_catalog.hrs_agent_metadata.hrs_variable_review.
3. Protect existing human review decisions by updating classification fields only for records that have not yet been reviewed.

In [0]:

from pyspark.sql import functions as F

review_table = "dev_catalog.hrs_agent_metadata.hrs_variable_review"

# ------------------------------------------------------------
# 1. Preliminary naming pattern for likely HRS variables.
# Longitudinal variable names (such as those used in the RAND HRS dataset)
# Visual Breakdown
#   [RS} - Variables that start with R (respondent) or S (spouse)
#   [0-9]{1,2} - Wave numbers from 0 to 99, 1 or two digits
#   [A-Z] - Core variable indicator. Must be followed by exactly one uppercase letter (A-Z)
#   [A-Z0-9]* : Remaining descriptors. Can be followed by zero or more uppercase letters or numbers.
# ------------------------------------------------------------
variable_pattern = r"^[RS][0-9]{1,2}[A-Z][A-Z0-9]*$"

# ------------------------------------------------------------
# 2. Display data before classification
# ------------------------------------------------------------

before = spark.table(review_table)

print("Records before classification:")
display(
    before.groupBy("review_status", "candidate_class")
    .count()
    .orderBy("review_status", "candidate_class")
)

# ------------------------------------------------------------
# 3. Classify only records that have not yet been classified.
#
# Safety:
# - Only Pending records are eligible.
# - Existing candidate_class values are preserved.
# - Existing review decisions are not overwritten.
# ------------------------------------------------------------

spark.sql(f"""
    UPDATE {review_table}
    SET
        candidate_class = CASE
            WHEN source_page BETWEEN 1 AND 52
                THEN 'Introductory page'
            WHEN candidate_token RLIKE '{variable_pattern}'
                THEN 'Likely variable'
            ELSE 'Needs review'
        END,

        classification_reason = CASE
            WHEN source_page BETWEEN 1 AND 52
                THEN 'Candidate found on an introductory Codebook page; excluded from preliminary variable classification'
            WHEN candidate_token RLIKE '{variable_pattern}'
                THEN 'Matches the preliminary variable naming pattern'
            ELSE 'Does not match the preliminary pattern; inspect Codebook evidence'
        END,

        classified_at = current_timestamp()

    WHERE review_status = 'Pending'
      AND candidate_class IS NULL
""")

# ------------------------------------------------------------
# 4. Validate the results.
# ------------------------------------------------------------

after = spark.table(review_table)

print("Records after classification:")
display(
    after.groupBy("review_status", "candidate_class")
    .count()
    .orderBy("review_status", "candidate_class")
)

# ------------------------------------------------------------
# 5. Display data after classification
# ------------------------------------------------------------

print("Records still awaiting initial classification:")
display(
    after.filter(
        (F.col("review_status") == "Pending")
        & F.col("candidate_class").isNull()
    ).select(
        "candidate_token",
        "source_page",
        "review_status"
    ).limit(50)
)


### Validate classifications

 This cell is intended to validate the classification counts and display examples from selected Codebook pages, including pages 1 and 2096.

 Validate classification counts and inspect examples provided for data review and confirmation.


In [0]:

from pyspark.sql import functions as F

review_table = "dev_catalog.hrs_agent_metadata.hrs_variable_review"

# Validate the number of records and distinct tokens per class.
display(
    spark.table(review_table)
    .groupBy("candidate_class")
    .agg(
        F.count("*").alias("candidate_record_count"),
        F.countDistinct("candidate_token").alias(
            "distinct_token_count"
        )
    )
    .orderBy("candidate_class")
)

# Inspect candidate examples from selected Codebook pages.
display(
    spark.table(review_table)
    .filter(F.col("source_page").isin(1, 2096))
    .select(
        "candidate_token",
        "source_page",
        "candidate_class",
        "classification_reason",
        "review_status",
        "context_excerpt"
    )
    .orderBy("source_page", "candidate_token")
    .limit(100)
)


### Summarize candidate frequency
Summarize how frequently candidate tokens occur and across how many pages.


In [0]:
%sql

SELECT
    candidate_token,
    candidate_class,
    COUNT(*) AS candidate_record_count,
    COUNT(DISTINCT source_page) AS distinct_page_count,
    MIN(source_page) AS first_page,
    MAX(source_page) AS last_page
FROM dev_catalog.hrs_agent_metadata.hrs_variable_review
GROUP BY
    candidate_token,
    candidate_class
ORDER BY
    candidate_record_count DESC,
    candidate_token
LIMIT 100;
